# Chương 2. Khoảng trống, tính dừng và phân rã chuỗi

Chương này gồm ba kỹ thuật: (1) điền khoảng trống có kiểm soát, (2) kiểm định và biến đổi để đạt tính dừng, (3) phân rã chuỗi. Kết quả phân rã được lưu lại làm đầu vào cho chương 3.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dmprep import config
from dmprep.data import load_main_series

config.set_seed()
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

RESULTS: dict = {}  # mọi score / p-value của chương này được lưu vào đây

In [2]:
series = load_main_series()
n = len(series)
print(series.name, n)

AAPL 5283


## 2.1 Xử lý khoảng trống và so sánh nội suy

Tạo nhân tạo 5% khoảng trống theo cơ chế **MCAR** (xác suất mất không phụ thuộc vào giá trị), rồi so sánh 4 phương pháp: forward fill, nội suy tuyến tính, đa thức bậc 2 và 3, cubic spline.

Đánh giá trên các vị trí bị xóa $\mathcal{M}$:

$$\text{MAE} = \frac{1}{|\mathcal M|}\sum_{t\in\mathcal M}|y_t-\hat y_t|, \qquad \text{RMSE} = \sqrt{\frac{1}{|\mathcal M|}\sum_{t\in\mathcal M}(y_t-\hat y_t)^2}$$

Kiểm định Ljung-Box với $H_0$: không có tự tương quan đến lag $h$:

$$Q = n(n+2)\sum_{k=1}^{h}\frac{\hat\rho_k^2}{n-k} \sim \chi^2_h$$

> TODO: viết lại phần lý thuyết bằng lời của bạn.

In [3]:
rng = np.random.default_rng(config.SEED)
y_true = series.reset_index(drop=True)          # index số nguyên: tránh lỗi nội suy đa thức trên DatetimeIndex
candidates = np.arange(1, len(y_true) - 1)      # không xóa điểm đầu / cuối
mask_idx = np.sort(rng.choice(candidates, size=int(0.05 * len(y_true)), replace=False))

y_missing = y_true.copy()
y_missing.iloc[mask_idx] = np.nan
print("Số điểm bị xóa:", len(mask_idx), f"({len(mask_idx) / len(y_true):.1%})")
RESULTS["gap_n_masked"] = len(mask_idx)

Số điểm bị xóa: 264 (5.0%)


In [4]:
# TODO: điền khuyết và đánh giá
# imputed = {
#     "ffill": y_missing.ffill(),
#     "linear": y_missing.interpolate("linear"),
#     "poly2": y_missing.interpolate("polynomial", order=2),
#     "poly3": y_missing.interpolate("polynomial", order=3),
#     "cubic_spline": y_missing.interpolate("spline", order=3),
# }
# from statsmodels.stats.diagnostic import acorr_ljungbox
# rows = []
# for name, y_hat in imputed.items():
#     err = y_true.iloc[mask_idx] - y_hat.iloc[mask_idx]
#     lb = acorr_ljungbox(..., lags=[10], return_df=True)   # chọn rõ chuỗi nào được kiểm định và giải thích
#     rows.append({"method": name, "MAE": err.abs().mean(), "RMSE": np.sqrt((err**2).mean()),
#                  "LB_stat": lb["lb_stat"].iloc[0], "LB_pvalue": lb["lb_pvalue"].iloc[0]})
# gap_table = pd.DataFrame(rows).set_index("method")
# RESULTS["gap_table"] = gap_table.to_dict()
# gap_table

**Phân tích kết quả.**

> TODO: phương pháp nào tốt nhất, vì sao (giá có tính liên tục cao nên tuyến tính thường tốt; đa thức bậc cao có thể dao động mạnh). Diễn giải p-value Ljung-Box.

## 2.2 Kiểm định tính dừng

| Kiểm định | $H_0$ | Kết luận dừng khi |
|---|---|---|
| ADF | có nghiệm đơn vị (không dừng) | p < 0.05 |
| KPSS | chuỗi dừng | p ≥ 0.05 |
| Phillips-Perron | có nghiệm đơn vị | p < 0.05 |

Nếu chuỗi không dừng, thử lần lượt: sai phân bậc 1 và 2, log, Box-Cox $y^{(\lambda)} = \frac{y^\lambda-1}{\lambda}$ ($\lambda\ne0$), và kết hợp (log rồi sai phân = log-return). Kiểm định lại sau mỗi biến đổi.

> TODO: viết lại phần lý thuyết bằng lời của bạn; giải thích vì sao dùng ADF và KPSS cùng lúc (hai giả thuyết ngược nhau).

In [5]:
# TODO: viết hàm kiểm định 3 test và lập bảng cho chuỗi gốc + từng biến đổi
# from statsmodels.tsa.stattools import adfuller, kpss
# from arch.unitroot import PhillipsPerron
# from scipy.stats import boxcox
#
# def stationarity_tests(x: pd.Series) -> dict:
#     x = x.dropna()
#     return {
#         "ADF_p": adfuller(x, autolag="AIC")[1],
#         "KPSS_p": kpss(x, regression="c", nlags="auto")[1],
#         "PP_p": PhillipsPerron(x).pvalue,
#     }
#
# transforms = {
#     "original": series,
#     "diff1": series.diff(),
#     "diff2": series.diff().diff(),
#     "log": np.log(series),
#     "boxcox": pd.Series(boxcox(series)[0], index=series.index),
#     "log_diff1": np.log(series).diff(),
# }
# stationarity_table = pd.DataFrame({k: stationarity_tests(v) for k, v in transforms.items()}).T
# RESULTS["stationarity"] = stationarity_table.to_dict()
# stationarity_table

**Phân tích kết quả.**

> TODO: chuỗi gốc không dừng; biến đổi nào đạt tính dừng theo cả 3 test. Lưu ý KPSS chỉ trả p-value trong khoảng [0.01, 0.1] (có cảnh báo).

## 2.3 Phân rã chuỗi thời gian

- **Additive:** $y_t = T_t + S_t + R_t$
- **Multiplicative:** $y_t = T_t \times S_t \times R_t$
- **STL** (Seasonal-Trend decomposition using LOESS): ước lượng $T_t, S_t$ bằng hồi quy cục bộ, bền vững với ngoại lai.

Đánh giá chất lượng bằng tỉ lệ phương sai phần dư $\text{Var}(R_t)/\text{Var}(y_t)$ (càng nhỏ, thành phần trend + seasonal giải thích càng nhiều).

**Chọn chu kỳ mùa vụ.** Dữ liệu phiên giao dịch: tuần ≈ 5 phiên, năm ≈ 252 phiên. Hãy thử và biện luận lựa chọn.

In [6]:
PERIOD = 252  # TODO: thử 5 và 252, biện luận

# TODO: phân rã additive, multiplicative (statsmodels.tsa.seasonal.seasonal_decompose) và STL (statsmodels.tsa.seasonal.STL)
# from statsmodels.tsa.seasonal import seasonal_decompose, STL
# y = np.log(series)  # gợi ý: phân rã trên log-giá
# stl = STL(y, period=PERIOD, robust=True).fit()
# resid_ratio = stl.resid.var() / y.var()
#
# Cuối mục: lưu chuỗi đã phân rã cho chương 3
# decomposed = pd.DataFrame({
#     "y": y, "trend": stl.trend, "seasonal": stl.seasonal, "resid": stl.resid,
#     "deseasonalized": y - stl.seasonal, "detrended": y - stl.trend,
# })
# decomposed.to_parquet(config.PROCESSED_DIR / "ts_decomposed.parquet")

**Phân tích kết quả.**

> TODO: so sánh 3 cách phân rã qua residual variance ratio và hình vẽ; thành phần mùa vụ có đáng kể không với giá cổ phiếu?

## Lưu kết quả

Toàn bộ con số của chương được ghi ra `data/processed/results/` để chương báo cáo tham chiếu.

In [7]:
config.save_results("02_ts_preprocessing", RESULTS)
RESULTS

{'gap_n_masked': 264}